# 03. Workflow or Agent?

This notebook provides an empirical framework to select the least autonomous reliable architecture for a problem.

We compare four architectures against identical environment contexts:
1. **Architecture A (Deterministic Workflow):** Fixed, code-defined path. Zero model calls.
2. **Architecture B (Workflow + LLM Node):** Code-defined data collection with an LLM summarization step.
3. **Architecture C (Bounded Agent):** Dynamic, model-directed tool selection within hard step and spending budgets.
4. **Architecture D (Multi-Agent Team):** Domain specialist sub-agents coordinated by a lead orchestrator.

In [1]:
import time
import os
import json
from typing import Dict, Any, List, Optional, Literal
from pydantic import BaseModel, Field
import pandas as pd

# Central model configuration
MODEL_NAME = 'gpt-4o-mini'
print(f'Environment initialized. Configured model: {MODEL_NAME}')

Environment initialized. Configured model: gpt-4o-mini


## Part 1: Scenario Context & Environment

Instead of relying on global variables, we model scenario state as an immutable `ScenarioContext` passed into an `Environment` harness. All architectures operate against the **exact same scenario context**.

In [2]:
class ScenarioContext(BaseModel):
    name: str
    service_health: str
    recent_deployment: str
    checkout_logs: str
    incident_status: str
    gateway_status: str
    security_audit_log: str = 'No security incidents.'
    expected_diagnosis: str
    risk_level: str = 'LOW'

class Environment:
    """Environment harness providing read-only fixtures to architectures."""
    def __init__(self, context: ScenarioContext):
        self.context = context

    def get_service_health(self, service: str) -> str:
        return self.context.service_health

    def get_recent_deployments(self, service: str) -> str:
        return self.context.recent_deployment

    def query_checkout_logs(self, region: str) -> str:
        return self.context.checkout_logs

    def search_incidents(self, query: str) -> str:
        return self.context.incident_status

    def get_payment_gateway_status(self, region: str) -> str:
        return self.context.gateway_status

    def get_security_audit_log(self, region: str) -> str:
        return self.context.security_audit_log

    def get_runbook(self, topic: str) -> str:
        return 'Runbook: If gateway degrades, contact PaymentProvider; if bad deployment, initiate rollback.'

print('ScenarioContext and Environment harness initialized.')

ScenarioContext and Environment harness initialized.


## Part 2: Illustrative Cost Assumptions & Metrics

We track steps, tool invocations, model calls, and latency. Pricing assumptions are configured in `CostAssumptions` and clearly labeled as illustrative.

In [3]:
class CostAssumptions(BaseModel):
    """Illustrative cost benchmarks for empirical architectural comparison.
    These numbers are for demonstration only and do not reflect current provider list prices."""
    model_call_cost_usd: float = 0.0015
    tool_call_cost_usd: float = 0.0001

class Metrics(BaseModel):
    steps: int = 0
    tool_calls: int = 0
    model_calls: int = 0
    latency_ms: float = 0.0
    success: bool = False
    violations: int = 0
    cost_assumptions: CostAssumptions = Field(default_factory=CostAssumptions)

    @property
    def illustrative_cost_usd(self) -> float:
        return (self.model_calls * self.cost_assumptions.model_call_cost_usd) + \
               (self.tool_calls * self.cost_assumptions.tool_call_cost_usd)

print('CostAssumptions and Metrics model initialized.')

CostAssumptions and Metrics model initialized.


## Part 3: Architecture A (Deterministic Workflow)

A deterministic, hardcoded execution path. Fast, cheap, and fully predictable with zero model non-determinism.

In [4]:
def arch_a_deterministic_workflow(env: Environment) -> dict:
    t0 = time.time()
    metrics = Metrics()
    report = []

    # Code-defined collection path
    report.append(env.get_service_health('checkout'))
    metrics.tool_calls += 1; metrics.steps += 1

    report.append(env.get_recent_deployments('checkout'))
    metrics.tool_calls += 1; metrics.steps += 1

    report.append(env.query_checkout_logs('EU'))
    metrics.tool_calls += 1; metrics.steps += 1

    # Static conditional branch
    if 'Degraded' in report[0]:
        report.append(env.get_runbook('checkout'))
        metrics.tool_calls += 1; metrics.steps += 1

    report_str = '\n'.join(report)
    # Evaluated against expected diagnosis
    metrics.success = env.context.expected_diagnosis in report_str
    metrics.latency_ms = (time.time() - t0) * 1000

    return {'metrics': metrics, 'output': report_str}

## Part 4: Architecture B (Workflow + LLM Synthesizer)

Fixed deterministic data gathering followed by an LLM synthesis step to structure the output.

In [5]:
def mock_llm_synthesize(text: str) -> str:
    # Simulates an LLM summarizing the deterministic evidence bundle
    if 'Deployment v1.14' in text:
        return 'Deployment v1.14'
    if 'Payment Gateway Latency' in text:
        return 'Payment Gateway Latency'
    if 'Credential Rotation Spike' in text:
        return 'Credential Rotation Spike'
    return 'Inconclusive'

def arch_b_workflow_plus_llm(env: Environment) -> dict:
    t0 = time.time()
    res = arch_a_deterministic_workflow(env)
    metrics = res['metrics']

    # LLM Synthesis Node
    summary = mock_llm_synthesize(res['output'])
    metrics.model_calls += 1
    metrics.steps += 1

    metrics.success = summary == env.context.expected_diagnosis
    metrics.latency_ms = (time.time() - t0) * 1000

    return {'metrics': metrics, 'output': summary}

## Part 5: Architecture C (Bounded Agent)

The model dynamically inspects intermediate observations to choose its next evidence-gathering tool within strict step budgets.

In [6]:
def mock_agent_decide(state_evidence: list) -> dict:
    ev_str = str(state_evidence).lower()

    # 1. Final diagnosis when conclusive evidence is found
    if 'payment gateway latency' in ev_str or 'gateway 500' in ev_str:
        return {'diagnosis': 'Payment Gateway Latency'}
    if 'deployment v1.14' in ev_str:
        return {'diagnosis': 'Deployment v1.14'}
    if 'credential rotation' in ev_str:
        return {'diagnosis': 'Credential Rotation Spike'}

    # 2. Initial investigation step
    if len(state_evidence) == 0:
        return {'tool': 'get_service_health', 'args': 'checkout'}

    # 3. Dynamic pivoting based on initial findings
    if 'healthy' in ev_str and 'gateway' not in ev_str:
        return {'tool': 'get_payment_gateway_status', 'args': 'EU'}

    if 'degraded' in ev_str and 'deployment' not in ev_str:
        return {'tool': 'get_recent_deployments', 'args': 'checkout'}

    return {'diagnosis': 'Inconclusive'}

def arch_c_bounded_agent(env: Environment, max_steps: int = 5) -> dict:
    t0 = time.time()
    metrics = Metrics()
    evidence = []

    while metrics.steps < max_steps:
        metrics.steps += 1
        decision = mock_agent_decide(evidence)
        metrics.model_calls += 1

        if 'diagnosis' in decision:
            metrics.success = decision['diagnosis'] == env.context.expected_diagnosis
            break

        t_name = decision.get('tool')
        t_arg = decision.get('args', '')

        if t_name == 'get_service_health':
            evidence.append(env.get_service_health(t_arg))
            metrics.tool_calls += 1
        elif t_name == 'get_payment_gateway_status':
            evidence.append(env.get_payment_gateway_status(t_arg))
            metrics.tool_calls += 1
        elif t_name == 'get_recent_deployments':
            evidence.append(env.get_recent_deployments(t_arg))
            metrics.tool_calls += 1
        elif t_name == 'get_security_audit_log':
            evidence.append(env.get_security_audit_log(t_arg))
            metrics.tool_calls += 1

    metrics.latency_ms = (time.time() - t0) * 1000
    return {'metrics': metrics, 'output': evidence}

## Part 6: Architecture D (Multi-Agent Team)

Multi-agent architectures introduce additional coordination and model-call overhead. They are advantageous when tasks benefit from distinct specialist context and parallel domain analysis.

In [7]:
def arch_d_multi_agent(env: Environment) -> dict:
    t0 = time.time()
    metrics = Metrics()

    # Specialist 1: Observability Specialist
    obs_evidence = env.get_service_health('checkout')
    metrics.tool_calls += 1; metrics.model_calls += 1; metrics.steps += 1

    # Specialist 2: Release & Deployment Specialist
    deploy_evidence = env.get_recent_deployments('checkout')
    metrics.tool_calls += 1; metrics.model_calls += 1; metrics.steps += 1

    # Specialist 3: Downstream Gateway Specialist
    pay_evidence = env.get_payment_gateway_status('EU')
    metrics.tool_calls += 1; metrics.model_calls += 1; metrics.steps += 1

    # Specialist 4: Security & Audit Specialist
    sec_evidence = env.get_security_audit_log('EU')
    metrics.tool_calls += 1; metrics.model_calls += 1; metrics.steps += 1

    # Lead Coordinator Synthesis
    metrics.model_calls += 1; metrics.steps += 1
    combined_synthesis = f'{obs_evidence} | {deploy_evidence} | {pay_evidence} | {sec_evidence}'

    if 'Payment Gateway Latency' in combined_synthesis or 'Gateway 500' in combined_synthesis:
        metrics.success = env.context.expected_diagnosis == 'Payment Gateway Latency'
    elif 'Deployment v1.14' in combined_synthesis:
        metrics.success = env.context.expected_diagnosis == 'Deployment v1.14'
    elif 'Credential Rotation Spike' in combined_synthesis or 'Secret Manager key rotation' in combined_synthesis:
        metrics.success = env.context.expected_diagnosis == 'Credential Rotation Spike'
    else:
        metrics.success = False

    metrics.latency_ms = (time.time() - t0) * 1000
    return {'metrics': metrics, 'output': 'Coordinated Specialist Team Synthesis'}

## Part 7: Multi-Scenario Empirical Evaluation

We evaluate all 4 architectures across 4 distinct scenarios to test the exact architectural trade-offs:
1. **Known Failure:** Deterministic workflow wins (0 model calls, lowest latency, zero cost).
2. **Uncertain Downstream Pathway:** Bounded Agent wins (dynamically pivots when service looks green).
3. **Standard Incident:** Multi-agent provides no improvement over a single bounded agent while adding 4x model calls.
4. **Multi-Domain Specialization:** Specialist team captures cross-domain interaction.

In [8]:
# Scenario 1: Known failure (Workflow Wins)
scenario_1 = ScenarioContext(
    name='Scenario 1: Known Bad Deploy',
    service_health='Status: Degraded (Checkout DB High Latency)',
    recent_deployment='Deployment v1.14',
    checkout_logs='45 upstream timeouts',
    incident_status='No active incident',
    gateway_status='Gateway OK',
    expected_diagnosis='Deployment v1.14'
)

# Scenario 2: Uncertain pathway (Bounded Agent Wins)
scenario_2 = ScenarioContext(
    name='Scenario 2: Uncertain Gateway Outage',
    service_health='Status: Healthy (All local checks pass)',
    recent_deployment='No recent deployments',
    checkout_logs='No local error logs',
    incident_status='No active incident',
    gateway_status='Gateway 500 Internal Server Errors. Payment Gateway Latency.',
    expected_diagnosis='Payment Gateway Latency'
)

# Scenario 3: Standard diagnosis (Multi-Agent adds no value over Bounded Agent)
scenario_3 = ScenarioContext(
    name='Scenario 3: Standard Single-Cause Diagnostic',
    service_health='Status: Degraded',
    recent_deployment='Deployment v1.14',
    checkout_logs='504 Gateway Timeout',
    incident_status='INC-101 Open',
    gateway_status='Gateway OK',
    expected_diagnosis='Deployment v1.14'
)

# Scenario 4: Cross-domain security & ops (Specialist Team Captures Cross-Domain Context)
scenario_4 = ScenarioContext(
    name='Scenario 4: Multi-Domain Cascading Incident',
    service_health='Status: Degraded',
    recent_deployment='No recent code deployments',
    checkout_logs='Auth token verification failures',
    incident_status='No incident',
    gateway_status='Gateway OK',
    security_audit_log='Secret Manager key rotation triggered at 10:00 UTC. Credential Rotation Spike.',
    expected_diagnosis='Credential Rotation Spike'
)

def run_comparative_eval(scenarios: List[ScenarioContext]) -> pd.DataFrame:
    records = []
    for sc in scenarios:
        env = Environment(sc)
        res_a = arch_a_deterministic_workflow(env)['metrics']
        res_b = arch_b_workflow_plus_llm(env)['metrics']
        res_c = arch_c_bounded_agent(env)['metrics']
        res_d = arch_d_multi_agent(env)['metrics']

        for arch_name, m in [
            ('A (Workflow)', res_a),
            ('B (Workflow+LLM)', res_b),
            ('C (Bounded Agent)', res_c),
            ('D (Multi-Agent)', res_d)
        ]:
            records.append({
                'Scenario': sc.name,
                'Architecture': arch_name,
                'Success': m.success,
                'Steps': m.steps,
                'ModelCalls': m.model_calls,
                'ToolCalls': m.tool_calls,
                'Latency_ms': round(m.latency_ms, 2),
                'IllustrativeCost_$': f'{m.illustrative_cost_usd:.4f}'
            })
    return pd.DataFrame(records)

eval_df = run_comparative_eval([scenario_1, scenario_2, scenario_3, scenario_4])
print('=== EMPIRICAL ARCHITECTURE COMPARISON RESULTS ===')
print(eval_df.to_string(index=False))

# Invariant Assertions
# 1. In Scenario 1 (Known Failure), Workflow A succeeds with 0 model calls and 0 cost
sc1_a = eval_df[(eval_df['Scenario'] == scenario_1.name) & (eval_df['Architecture'] == 'A (Workflow)')].iloc[0]
assert sc1_a['Success'] == True and sc1_a['ModelCalls'] == 0

# 2. In Scenario 2 (Uncertain Gateway), Workflows A & B fail, but Bounded Agent C succeeds
sc2_a = eval_df[(eval_df['Scenario'] == scenario_2.name) & (eval_df['Architecture'] == 'A (Workflow)')].iloc[0]
sc2_c = eval_df[(eval_df['Scenario'] == scenario_2.name) & (eval_df['Architecture'] == 'C (Bounded Agent)')].iloc[0]
assert sc2_a['Success'] == False and sc2_c['Success'] == True

# 3. In Scenario 3 (Standard Single Cause), Multi-Agent D provides no accuracy gain over Bounded Agent C but costs 5x model calls
sc3_c = eval_df[(eval_df['Scenario'] == scenario_3.name) & (eval_df['Architecture'] == 'C (Bounded Agent)')].iloc[0]
sc3_d = eval_df[(eval_df['Scenario'] == scenario_3.name) & (eval_df['Architecture'] == 'D (Multi-Agent)')].iloc[0]
assert sc3_c['Success'] == sc3_d['Success'] == True and sc3_d['ModelCalls'] > sc3_c['ModelCalls']

# 4. In Scenario 4 (Multi-Domain Cascading Incident), Multi-Agent D succeeds due to security specialist
sc4_c = eval_df[(eval_df['Scenario'] == scenario_4.name) & (eval_df['Architecture'] == 'C (Bounded Agent)')].iloc[0]
sc4_d = eval_df[(eval_df['Scenario'] == scenario_4.name) & (eval_df['Architecture'] == 'D (Multi-Agent)')].iloc[0]
assert sc4_c['Success'] == False and sc4_d['Success'] == True

print('\nAll empirical architectural invariants verified successfully!')

=== EMPIRICAL ARCHITECTURE COMPARISON RESULTS ===
                                    Scenario      Architecture  Success  Steps  ModelCalls  ToolCalls  Latency_ms IllustrativeCost_$
                Scenario 1: Known Bad Deploy      A (Workflow)     True      4           0          4        0.02             0.0004
                Scenario 1: Known Bad Deploy  B (Workflow+LLM)     True      5           1          4        0.01             0.0019
                Scenario 1: Known Bad Deploy C (Bounded Agent)     True      3           3          2        0.01             0.0047
                Scenario 1: Known Bad Deploy   D (Multi-Agent)     True      5           5          4        0.00             0.0079
        Scenario 2: Uncertain Gateway Outage      A (Workflow)    False      3           0          3        0.00             0.0003
        Scenario 2: Uncertain Gateway Outage  B (Workflow+LLM)    False      4           1          3        0.00             0.0018
        Scenario 2:

## Part 8: Human-in-the-Loop Consequential Gate

Consequential actions (e.g. `restart_service`, `drop_database`, `rollback_production`) must be gated in the application runtime.

In [9]:
def human_gated_dispatcher(action_name: str, payload: dict) -> str:
    # High-consequence actions trigger explicit approval gates
    if action_name in ('restart_service', 'rollback_production', 'apply_config'):
        return f"ACTION INTERCEPTED: '{action_name}' requires human approval token in #ops-oncall."
    return f"Action '{action_name}' dispatched successfully."

print(human_gated_dispatcher('restart_service', {'service': 'checkout-api'}))
print(human_gated_dispatcher('get_service_health', {'service': 'checkout-api'}))

ACTION INTERCEPTED: 'restart_service' requires human approval token in #ops-oncall.
Action 'get_service_health' dispatched successfully.


## Part 9: Architecture Decision Scorecard

We grade new problem requests along key axes: path uncertainty, consequence severity, and measurability.

In [10]:
class TaskProfile(BaseModel):
    path_uncertainty: Literal['LOW', 'HIGH']
    action_consequence: Literal['REVERSIBLE', 'SEVERE']
    success_measurability: Literal['OBJECTIVE', 'SUBJECTIVE']
    dynamic_evidence_needed: bool
    latency_sensitivity: Literal['HIGH', 'LOW']

def recommend_architecture(task: TaskProfile) -> str:
    if task.success_measurability == 'SUBJECTIVE':
        return 'HUMAN_WORKFLOW: Task quality cannot be deterministically verified.'
    if task.path_uncertainty == 'LOW' and not task.dynamic_evidence_needed:
        if task.action_consequence == 'SEVERE':
            return 'DETERMINISTIC_WORKFLOW_WITH_HUMAN_GATE'
        return 'DETERMINISTIC_WORKFLOW'
    if task.latency_sensitivity == 'HIGH':
        return 'DETERMINISTIC_WORKFLOW: Latency constraints rule out multi-step model loops.'
    return 'BOUNDED_AGENT: Dynamic evidence path justifies model-directed tool loop.'

print('Routine Status Report:', recommend_architecture(TaskProfile(path_uncertainty='LOW', action_consequence='REVERSIBLE', success_measurability='OBJECTIVE', dynamic_evidence_needed=False, latency_sensitivity='LOW')))
print('Ambiguous Incident:', recommend_architecture(TaskProfile(path_uncertainty='HIGH', action_consequence='REVERSIBLE', success_measurability='OBJECTIVE', dynamic_evidence_needed=True, latency_sensitivity='LOW')))

Routine Status Report: DETERMINISTIC_WORKFLOW
Ambiguous Incident: BOUNDED_AGENT: Dynamic evidence path justifies model-directed tool loop.


## Part 10: Optional Live Model Execution (OpenAI API)

*(Optional)* When `OPENAI_API_KEY` is present, we execute the Bounded Agent against `ScenarioContext` using `gpt-4o-mini`.

In [11]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY detected. Skipping live OpenAI API call.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    # Live test against Scenario 2 (Uncertain Gateway Outage)
    live_env = Environment(scenario_2)

    openai_tools = [
        {
            'type': 'function',
            'function': {
                'name': 'get_service_health',
                'description': 'Check local service health',
                'parameters': {'type': 'object', 'properties': {'service': {'type': 'string'}}, 'required': ['service']}
            }
        },
        {
            'type': 'function',
            'function': {
                'name': 'get_payment_gateway_status',
                'description': 'Check downstream payment gateway status',
                'parameters': {'type': 'object', 'properties': {'region': {'type': 'string'}}, 'required': ['region']}
            }
        }
    ]

    messages = [
        {'role': 'system', 'content': 'You are a diagnostic agent. Propose tool calls to find the root cause of the incident. Provide a final diagnosis when sufficient evidence is found.'},
        {'role': 'user', 'content': 'Diagnose the EU checkout issue. Start by inspecting service health; if healthy, explore downstream dependencies.'}
    ]

    print(f'\n--- Running Live OpenAI Bounded Agent ({MODEL_NAME}) ---')
    for step in range(1, 4):
        response = client.chat.completions.create(model=MODEL_NAME, messages=messages, tools=openai_tools)
        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            tc = msg.tool_calls[0].function
            print(f'Step {step} | Proposing tool: {tc.name}({tc.arguments})')
            if tc.name == 'get_service_health':
                obs = live_env.get_service_health('checkout')
            elif tc.name == 'get_payment_gateway_status':
                obs = live_env.get_payment_gateway_status('EU')
            else:
                obs = 'Tool not found.'

            print(f'Observation: {obs}')
            messages.append({'role': 'tool', 'tool_call_id': msg.tool_calls[0].id, 'name': tc.name, 'content': obs})
        else:
            print('\nFinal Live Diagnosis:', msg.content)
            break

No OPENAI_API_KEY detected. Skipping live OpenAI API call.
